# 2. Govern model access

**Question:** Can two consumers share one authenticated model gateway with separate counters?

The existing APIM add-on authenticates subscription keys and uses managed identity to call the model. Its token-limit counter uses subscription ID, not the caller-controlled `x-agent-name` label. This route calls the model directly; it is not a published-agent invocation.

The default examples below are **synthetic**. A 429 demonstrates a rejected request but, by itself, does not identify whether APIM or the backend imposed the quota. A fixed number of calls does not guarantee a 429.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/governance').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from IPython.display import Markdown, display
from src.governance.evidence import synthetic_estate, markdown_table

usage = [row.model_dump(mode='json') for row in synthetic_estate().evidence if row.category == 'usage']
display(Markdown('**Synthetic gateway responses; not live traffic or billing evidence.**'))
display(Markdown(markdown_table(usage)))

## Explicit inference boundary

`invoke_gateway` reuses `scripts.demo.gateway_demo.call`. It requires `allow_inference=True` and checks the Commercial Azure CLI context. Keys come only from the ignored `.env`; returned evidence excludes request/response bodies, keys, and conversation content.

This cell issues exactly one request per consumer when enabled. Calls can incur charges and consume token limits. It neither changes the APIM policy nor retries a failed request. The table displays HTTP status, total/remaining tokens when returned, latency, and a request identifier when available. Missing usage stays null, not zero.

In [ ]:
ALLOW_PAID_INFERENCE = False

if ALLOW_PAID_INFERENCE:
    from dotenv import load_dotenv
    from src.governance.live import invoke_gateway
    load_dotenv(ROOT / '.env')
    results = [invoke_gateway(consumer, allow_inference=True) for consumer in ('asset-agent', 'partner-app')]
    display(Markdown(markdown_table([row.model_dump(mode='json') for row in results])))
else:
    print('Paid inference disabled. No model requests sent.')

## What this evidence establishes

- **401:** authentication failed; no claim about the model's health.
- **200:** the request was accepted; it may still have a filtered completion or empty output.
- **429:** a quota rejected the request; correlate gateway/backend evidence before attributing the cause.
- **Remaining-token header:** a returned counter observation, not the tenant's entire budget.

Aggregate request logs and token-metric queries are defined in [gateway_queries.kql](../../scripts/demo/gateway_queries.kql). Gateway metrics and logs are separate from Agent 365 activity and risk.